In [0]:
from pyspark.sql.functions import col, round, current_timestamp, when

# ── READ FROM BRONZE ─────────────────────────────────
bronze_df = spark.table("bronze.exchange_rates")

# ── TRANSFORM ────────────────────────────────────────
def transform_to_silver(df):
    silver_df = df \
        .filter(col("rate") > 0) \
        .withColumn("rate", round(col("rate"), 6)) \
        .withColumn("rate_vs_usd", round(1 / col("rate"), 6)) \
        .withColumn("processed_at", current_timestamp()) \
        .withColumn("rate_category",
            when(col("rate") < 1, "stronger_than_usd")
            .when(col("rate") == 1, "pegged_to_usd")
            .otherwise("weaker_than_usd")
        ) \
        .select(
            "currency",
            "base",
            "rate",
            "rate_vs_usd",
            "rate_category",
            "ingested_at",
            "processed_at"
        )
    return silver_df

silver_df = transform_to_silver(bronze_df)

# ── SAVE TO SILVER ───────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

silver_df.writeTo("silver.exchange_rates") \
    .using("delta") \
    .createOrReplace()

print(f"Silver table saved: {silver_df.count()} rows")

# ── VERIFY ───────────────────────────────────────────
spark.sql("SELECT * FROM silver.exchange_rates LIMIT 10").show()